In [1]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
import gradio as gr

In [2]:
db_name = "vector_db_mxbai"
embeddings = OllamaEmbeddings(model="mxbai-embed-large")
MODEL = "llama3.2:latest"
collection_name = "insurellm_mxbai"
vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings, collection_name=collection_name,)
print(f"Vector store initialized with database: {vectorstore._collection.count()}")


Vector store initialized with database: 413


In [3]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 15})
print(f"Retriever initialized with vector store: {vectorstore._collection.count()}")
llm = ChatOllama(model=MODEL,temperature= 0)

Retriever initialized with vector store: 413


In [4]:
retriever.invoke("what is health llm and leifellm")

[Document(id='7fdacfa3-2046-4c70-882e-8356da0a1574', metadata={'source': 'knowledge-base\\products\\Healthllm.md', 'doc_type': 'products'}, page_content="Healthllm represents Insurellm's commitment to transforming health insurance through technology that improves outcomes for insurers, providers, and members alike. Join us in building the future of health insurance!"),
 Document(id='e682c409-e5ea-4193-9006-6dc2dae64672', metadata={'source': 'knowledge-base\\products\\Healthllm.md', 'doc_type': 'products'}, page_content="# Product Summary\n\n# Healthllm\n\n## Summary\n\nHealthllm is Insurellm's comprehensive health insurance platform that empowers insurance providers to deliver modern, personalized health coverage. By combining advanced AI technology with healthcare data analytics, Healthllm streamlines every aspect of health insurance operations—from plan design and enrollment to claims processing and member engagement. Built for the complexities of the healthcare industry, Healthllm h

In [5]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [7]:
from langchain_core.messages import SystemMessage, HumanMessage

def chat(message, history):
    # 1. Fetch relevant documents
    docs = retriever.invoke(message)
    
    # 2. Convert the list of documents into a clean string block
    # (Or use `", ".join([repr(d) for d in docs])` if you strictly want the raw object string format)
    context_str = "\n\n".join([doc.page_content for doc in docs])
    
    # 3. Format the system prompt string
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context_str)
    
    # 4. Construct the messages using official LangChain message classes
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=message)
    ]
    
    print(f"Messages prepared for LLM: {messages}")
    
    try:
        # 5. Invoke the LLM (returns an AIMessage object)
        response_message = llm.invoke(messages)
        print(f"LLM response object: {response_message}")
        
        # 6. Extract the actual text string out of the AIMessage object
        response = response_message.content
        
    except Exception as e:
        response = f"An error occurred: {e}"
        print(response)
        
    return response


In [ ]:
gr.ChatInterface(fn=chat).launch()

c:\Users\neerajj\Usersneerajjllm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Messages prepared for LLM: [SystemMessage(content='\nYou are a knowledgeable, friendly assistant representing the company Insurellm.\nYou are chatting with a user about Insurellm.\nIf relevant, use the given context to answer any question.\nIf you don\'t know the answer, say so.\nContext:\n3. **Platform Updates:**\n   - Quarterly feature releases\n   - Monthly security patches and improvements\n   - Advance notice of updates (minimum 14 days)\n   - Release notes and user documentation\n\n4. **Account Management:**\n   - Named customer success manager\n   - Semi-annual business review meetings\n   - Usage analytics and optimization recommendations\n   - Assistance with tier upgrade when volume grows\n\n5. **Compliance Support:**\n   - Annual compliance review and audit support\n   - Regulatory update notifications\n   - State insurance department filing assistance\n\n---\n\n**Signatures:**\n\n_________________________________\n**Michael Torres**\n**Title**: Chief Revenue Officer\n**Insu